# tpt-soma Arrow Flight smoke test

Validates the `pyarrow.flight` client against the tpt-soma Flight service. A
capability token is required (TM-02): pass it as an `authorization` header on
every call.

1. Start the stack: `docker compose -f deploy/docker-compose.yml up -d`
2. Issue a token:

```bash
cargo run -p tpt-soma-api --bin admin -- gen-key
cargo run -p tpt-soma-api --bin admin -- issue \
  --subject researcher-1 --resource-class genomic_variant \
  --action read --cohort '*' --key dev-keys/signing_key.bin
```

3. Paste the printed JSON token into `TOKEN` below.


In [ ]:
import os
import pyarrow as pa
import pyarrow.flight as flight

FLIGHT_URL = os.environ.get("TPT_FLIGHT_URL", "grpc://localhost:8815")
TOKEN = os.environ.get("TPT_TOKEN", "PASTE_TOKEN_HERE")
SAMPLE_ID = os.environ.get("TPT_SAMPLE_ID", "00000000-0000-0000-0000-000000000000")

client = flight.connect(FLIGHT_URL)
options = flight.FlightCallOptions(
    headers=[(b"authorization", ("Bearer " + TOKEN).encode("utf-8"))]
)
command = f"variants:{SAMPLE_ID}"

## 1. Unauthenticated request must be rejected

If the server does **not** reject this, Flight authentication is broken.

In [ ]:
try:
    client.get_flight_info(
        flight.FlightDescriptor.for_command(command.encode("utf-8"))
    )
    raise AssertionError("unauthenticated request was NOT rejected")
except flight.FlightUnauthenticatedError:
    print("OK: unauthenticated request rejected")
except Exception as exc:  # noqa: BLE001
    print(f"OK: unauthenticated request rejected ({type(exc).__name__})")

## 2. Authorized request

Fetches variants for the sample and prints the Arrow schema and row count.

In [ ]:
descriptor = flight.FlightDescriptor.for_command(command.encode("utf-8"))
info = client.get_flight_info(descriptor, options=options)
ticket = info.endpoints[0].ticket

reader = client.do_get(ticket, options=options)
batches = []
try:
    while True:
        batches.append(reader.read_chunk().data)
except StopIteration:
    pass

nrows = sum(b.num_rows for b in batches)
print(f"OK: {len(batches)} batch(es), {nrows} row(s) for '{command}'")
for batch in batches:
    display(batch)